# Notebook 01 — Quality Control and Cell Filtering

**Project:** RetinoblastomaAtlas  
**Input:** `data/processed/01_merged_raw.h5ad`  
**Output:** `data/processed/02_qc_filtered.h5ad`, QC figures and tables

---

## Scientific Background

Single-cell RNA-seq data contain a mixture of high-quality cells and technical artifacts that must be removed before biological interpretation. The three principal artifact classes are:

1. **Dead / dying cells** — characterised by high mitochondrial gene fraction (mt%) because cytoplasmic mRNA is lost through damaged membranes while mitochondrial mRNA is retained. Inclusion of dying cells inflates technical variance and masks real cell-type differences.

2. **Empty droplets / ambient RNA** — GEMs (gel-bead-in-emulsions) that captured no real cell but contain floating mRNA from the solution. These appear as cells with very low UMI counts.

3. **Doublets** — two cells captured in the same GEM, mis-identified as one high-count cell. In retinoblastoma data, doublets create spurious "hybrid" populations that could be misannotated as transitional states — directly confounding the cone precursor trajectory analysis.

## Why Per-Sample Adaptive Thresholding?

Fixed cut-offs (e.g., mt% < 20%, nGenes > 300) applied uniformly to all samples can over-filter samples with genuinely low complexity (e.g., small tumour cell populations) or under-filter samples with poor cell viability. The **Median Absolute Deviation (MAD)** approach adapts to each sample's distribution, making it robust to the different sequencing depths and cell compositions between GSE168434 and GSE249995.

**References:**  
- Young MD, Behjati S. SoupX. *GigaScience* 2020. https://doi.org/10.1093/gigascience/giaa151  
- Wolock SL et al. Scrublet. *Cell Syst* 2019. https://doi.org/10.1016/j.cels.2018.11.005

## Parameters

In [1]:
# --- QC Parameters ---
MT_THRESHOLD      = 20.0   # Maximum mitochondrial % per cell
MIN_GENES         = 300    # Hard lower bound on genes detected per cell
MIN_COUNTS        = 500    # Hard lower bound on UMI counts per cell
N_MADS            = 5.0    # Number of MADs for per-sample outlier detection
DOUBLET_THRESHOLD = 0.25   # Scrublet score above which a cell is called a doublet
MIN_CELLS_GENE    = 20     # Minimum cells a gene must be expressed in

## Setup

In [2]:
from __future__ import annotations

import warnings
from pathlib import Path

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scrublet as scr
from scipy.sparse import issparse

warnings.filterwarnings("ignore")
sc.settings.verbosity = 1

ROOT     = Path("..").resolve()
IN_H5AD  = ROOT / "data" / "processed" / "01_merged_raw.h5ad"
OUT_H5AD = ROOT / "data" / "processed" / "02_qc_filtered.h5ad"
FIG_DIR  = ROOT / "results" / "figures"
TAB_DIR  = ROOT / "results" / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input:  {IN_H5AD}")
print(f"Output: {OUT_H5AD}")

Input:  /mnt/h/Projects/scRNA-Seq/RetinoblastomaAtlas/data/processed/01_merged_raw.h5ad
Output: /mnt/h/Projects/scRNA-Seq/RetinoblastomaAtlas/data/processed/02_qc_filtered.h5ad


## Step 1 — Load Raw Atlas

In [3]:
print("Step 1 — Loading merged raw atlas")
adata = sc.read_h5ad(IN_H5AD)
print(f"  Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")
print(f"  Samples: {adata.obs['sample_id'].nunique()}")
print(f"  Datasets: {adata.obs['dataset'].value_counts().to_dict()}")

Step 1 — Loading merged raw atlas
  Loaded: 141,880 cells × 61,108 genes
  Samples: 14
  Datasets: {'GSE168434': 91572, 'GSE249995': 50308}


## Step 2 — Compute Per-Cell QC Metrics

Scanpy's `calculate_qc_metrics()` computes for each cell:
- `n_genes_by_counts`: number of genes with >0 counts (complexity)
- `total_counts`: total UMI count (library size)
- `pct_counts_mt`: percentage of counts from mitochondrial genes (cell viability proxy)

The mitochondrial fraction is the primary indicator of cell viability. As a cell lyses, cytoplasmic mRNA is lost faster than mitochondria-enclosed mRNA, causing the MT fraction to rise sharply.

In [4]:
print("Step 2 — Computing per-cell QC metrics")
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True
)

print(f"  Median genes/cell:   {adata.obs['n_genes_by_counts'].median():.0f}")
print(f"  Median counts/cell:  {adata.obs['total_counts'].median():.0f}")
print(f"  Median mt%:          {adata.obs['pct_counts_mt'].median():.1f}%")

# Save raw QC table
qc_df = adata.obs[["sample_id", "dataset", "n_genes_by_counts", "total_counts", "pct_counts_mt"]].copy()
qc_df.to_csv(TAB_DIR / "qc_metrics_per_cell_raw.csv")
print(f"  Saved pre-filter QC table")

Step 2 — Computing per-cell QC metrics
  Median genes/cell:   2506
  Median counts/cell:  6454
  Median mt%:          2.4%
  Saved pre-filter QC table


### QC Violin Plots (Before Filtering)

These plots reveal batch effects in QC metrics. Large differences in sequencing depth or cell viability across samples would indicate technical rather than biological variation — important context for interpreting downstream batch correction.

In [7]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))
metrics = [
    ("n_genes_by_counts", "Genes detected per cell"),
    ("total_counts",      "Total UMI counts per cell"),
    ("pct_counts_mt",     "Mitochondrial gene fraction (%)"),
]
for ax, (metric, title) in zip(axes, metrics):
    sc.pl.violin(adata, metric, groupby="sample_id", rotation=45, show=False, ax=ax, size=0.3)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel("")
plt.suptitle("QC distributions per sample (before filtering)", fontsize=12, y=1.01)
plt.tight_layout()
fig.savefig(FIG_DIR / "qc_violin_before_filter.png", dpi=200, bbox_inches="tight")
plt.show()

### MT% vs. Total Counts Scatter

The classic 'hockey stick' pattern (high mt% at low counts) confirms that dying cells are being correctly identified. The log-scale x-axis reveals the bimodal distribution of library sizes separating real cells from empty droplets.

In [8]:
fig, ax = plt.subplots(figsize=(8, 6))
samples  = adata.obs["sample_id"].astype("category").cat.categories
palette  = plt.cm.tab20(np.linspace(0, 1, len(samples)))
for sample, color in zip(samples, palette):
    mask = adata.obs["sample_id"] == sample
    ax.scatter(
        adata.obs.loc[mask, "total_counts"],
        adata.obs.loc[mask, "pct_counts_mt"],
        s=1, alpha=0.3, color=color, label=sample, rasterized=True,
    )
ax.set_xlabel("Total UMI counts (log scale)", fontsize=11)
ax.set_ylabel("Mitochondrial %", fontsize=11)
ax.set_xscale("log")
ax.axhline(MT_THRESHOLD, color="red", linestyle="--", linewidth=1, label=f"MT threshold ({MT_THRESHOLD}%)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", markerscale=5, fontsize=7, frameon=False)
ax.set_title("QC scatter: total counts vs mitochondrial fraction", fontsize=10)
plt.tight_layout()
fig.savefig(FIG_DIR / "qc_scatter_mt_vs_counts.png", dpi=200, bbox_inches="tight")
plt.show()

## Step 3 — Doublet Detection (Scrublet, Per Sample)

**Why run Scrublet per sample?** Scrublet simulates doublets by combining pairs of observed cells. If run on the merged atlas, the expected doublet neighbourhood is dominated by the most abundant cell types and the per-sample expected doublet rate (which depends on cell loading density) cannot be set correctly. Running per-sample avoids both issues.

Scrublet score interpretation:
- **Singlet peak**: near 0 (most cells)
- **Doublet peak**: higher scores, ideally clearly separated from singlets
- **Manual threshold**: if Scrublet's automatic threshold is <0.15, we fall back to 0.25

In [9]:
def run_scrublet(adata_sample, sample_id, expected_doublet_rate=0.06, threshold=0.25):
    counts = adata_sample.X
    if issparse(counts):
        counts = counts.toarray()
    scrub = scr.Scrublet(counts, expected_doublet_rate=expected_doublet_rate)
    doublet_scores, predicted_doublets = scrub.scrub_doublets(
        min_counts=2, min_cells=3, n_prin_comps=30, verbose=False
    )
    if scrub.threshold_ < 0.15:
        print(f"    {sample_id}: Scrublet threshold {scrub.threshold_:.3f} very low; using manual {threshold}")
        predicted_doublets = doublet_scores >= threshold
    n_doublets = predicted_doublets.sum()
    pct = 100 * n_doublets / len(doublet_scores)
    print(f"    {sample_id}: {n_doublets} doublets ({pct:.1f}%)")
    return (
        pd.Series(doublet_scores,    index=adata_sample.obs_names, name="doublet_score"),
        pd.Series(predicted_doublets, index=adata_sample.obs_names, name="is_doublet"),
    )


print("Step 3 — Doublet detection (Scrublet, per sample)")
all_scores, all_predict = [], []
for sid in adata.obs["sample_id"].astype("category").cat.categories:
    mask = adata.obs["sample_id"] == sid
    scores, predicted = run_scrublet(
        adata[mask].copy(), sid, threshold=DOUBLET_THRESHOLD
    )
    all_scores.append(scores)
    all_predict.append(predicted)

adata.obs["doublet_score"] = pd.concat(all_scores).reindex(adata.obs_names)
adata.obs["is_doublet"]    = pd.concat(all_predict).reindex(adata.obs_names)

n_doublets = adata.obs["is_doublet"].sum()
print(f"  Total predicted doublets: {n_doublets:,} ({100*n_doublets/adata.n_obs:.1f}%)")

Step 3 — Doublet detection (Scrublet, per sample)
    GSM5139852_RB01_rep1: 0 doublets (0.0%)
    GSM5139853_RB01_rep2: 0 doublets (0.0%)
    GSM5139854_RB02_rep1: 2 doublets (0.0%)
    GSM5139855_RB02_rep2: 2 doublets (0.1%)
    GSM5139856_RB03_rep1: 3 doublets (0.0%)
    GSM5139857_RB03_rep2: 2 doublets (0.0%)
    GSM5139858_RB04: 0 doublets (0.0%)
    GSM5139859_RB05: 0 doublets (0.0%)
    GSM5139860_RB06: 0 doublets (0.0%)
    GSM5139861_RB07: 0 doublets (0.0%)
    S1_in1: 4 doublets (0.0%)
    S2_in2: 0 doublets (0.0%)
    S3_ex1: 0 doublets (0.0%)
    S4_ex2: 0 doublets (0.0%)
  Total predicted doublets: 13 (0.0%)


In [10]:
# Doublet score histogram — expect bimodal distribution
fig, ax = plt.subplots(figsize=(8, 5))
doublets = adata.obs["doublet_score"][adata.obs["is_doublet"] == True]
singlets = adata.obs["doublet_score"][adata.obs["is_doublet"] == False]
ax.hist(singlets, bins=100, alpha=0.7, color="#4C9BE8", label="Singlet")
ax.hist(doublets, bins=100, alpha=0.7, color="#E84C4C", label="Doublet")
ax.axvline(DOUBLET_THRESHOLD, color="black", linestyle="--", linewidth=1.5,
           label=f"Threshold ({DOUBLET_THRESHOLD})")
ax.set_xlabel("Scrublet doublet score", fontsize=11)
ax.set_ylabel("Number of cells", fontsize=11)
ax.set_title("Doublet score distribution (Scrublet)\nExpect bimodal: singlet peak near 0, doublet peak at higher scores", fontsize=10)
ax.legend(fontsize=10)
plt.tight_layout()
fig.savefig(FIG_DIR / "doublet_scores_histogram.png", dpi=200, bbox_inches="tight")
plt.show()

## Step 4 — Per-Sample MAD-Based Outlier Filtering

The MAD (Median Absolute Deviation) approach is more robust than mean ± n×SD for scRNA-seq count distributions, which have heavy tails. A cell is flagged as an outlier if its metric is more than `N_MADS` MADs from the sample median.

Hard lower bounds (`MIN_GENES`, `MIN_COUNTS`) are applied as absolute floors, regardless of the adaptive threshold, to catch obvious empty droplets.

In [11]:
def mad_threshold(series: pd.Series, n_mads: float = 5.0):
    median = np.median(series)
    mad    = np.median(np.abs(series - median))
    return median - n_mads * mad, median + n_mads * mad


print(f"Step 4 — Per-sample MAD filtering (n_mads={N_MADS})")
keep_mask    = pd.Series(True, index=adata.obs_names)
summary_rows = []

for sid in adata.obs["sample_id"].astype("category").cat.categories:
    mask = adata.obs["sample_id"] == sid
    sub  = adata.obs.loc[mask]

    lo_g, hi_g = mad_threshold(sub["n_genes_by_counts"], N_MADS)
    lo_c, hi_c = mad_threshold(sub["total_counts"],      N_MADS)
    lo_g = max(lo_g, MIN_GENES)
    lo_c = max(lo_c, MIN_COUNTS)

    bad = (
        (sub["n_genes_by_counts"] < lo_g) | (sub["n_genes_by_counts"] > hi_g) |
        (sub["total_counts"]      < lo_c) | (sub["total_counts"]      > hi_c)
    )
    keep_mask.loc[sub.index[bad]] = False
    summary_rows.append({
        "sample_id": sid, "n_cells_before": mask.sum(),
        "lo_genes": lo_g, "hi_genes": hi_g,
        "lo_counts": lo_c, "hi_counts": hi_c,
        "cells_failed": int(bad.sum()),
    })
    print(f"  {sid}: genes [{lo_g:.0f}, {hi_g:.0f}], counts [{lo_c:.0f}, {hi_c:.0f}] → {bad.sum()} cells removed")

print(f"\nStep 5 — Global MT% filter: mt% < {MT_THRESHOLD}%")
high_mt    = adata.obs["pct_counts_mt"] >= MT_THRESHOLD
keep_mask  = keep_mask & (~high_mt)
print(f"  Cells with mt% ≥ {MT_THRESHOLD}: {high_mt.sum():,}")

print("Step 6 — Removing predicted doublets")
keep_mask = keep_mask & (~adata.obs["is_doublet"].astype(bool))

Step 4 — Per-sample MAD filtering (n_mads=5.0)
  GSM5139852_RB01_rep1: genes [300, 5187], counts [500, 18112] → 108 cells removed
  GSM5139853_RB01_rep2: genes [300, 5044], counts [500, 17335] → 94 cells removed
  GSM5139854_RB02_rep1: genes [300, 4027], counts [500, 10915] → 325 cells removed
  GSM5139855_RB02_rep2: genes [300, 3981], counts [500, 10420] → 167 cells removed
  GSM5139856_RB03_rep1: genes [300, 4918], counts [500, 13741] → 438 cells removed
  GSM5139857_RB03_rep2: genes [300, 4294], counts [500, 12262] → 554 cells removed
  GSM5139858_RB04: genes [300, 6417], counts [500, 19375] → 280 cells removed
  GSM5139859_RB05: genes [300, 5911], counts [500, 19693] → 404 cells removed
  GSM5139860_RB06: genes [300, 5667], counts [500, 16793] → 228 cells removed
  GSM5139861_RB07: genes [300, 5316], counts [500, 16133] → 246 cells removed
  S1_in1: genes [300, 10600], counts [500, 46116] → 215 cells removed
  S2_in2: genes [300, 8431], counts [500, 30055] → 542 cells removed
  S3_

## Step 5 — Apply Filters and Review Results

In [12]:
adata_filt = adata[keep_mask].copy()
n_removed  = adata.n_obs - adata_filt.n_obs

print(f"  Cells before QC : {adata.n_obs:>8,}")
print(f"  Cells removed   : {n_removed:>8,} ({100*n_removed/adata.n_obs:.1f}%)")
print(f"  Cells retained  : {adata_filt.n_obs:>8,}")

# Remove genes expressed in <20 cells
print(f"\n  Removing genes expressed in <{MIN_CELLS_GENE} cells...")
sc.pp.filter_genes(adata_filt, min_cells=MIN_CELLS_GENE)
print(f"  Genes retained: {adata_filt.n_vars:,}")

  Cells before QC :  141,880
  Cells removed   :    5,681 (4.0%)
  Cells retained  :  136,199

  Removing genes expressed in <20 cells...
  Genes retained: 28,306


In [13]:
# QC violins after filtering
fig, axes = plt.subplots(3, 1, figsize=(14, 12))
metrics = [
    ("n_genes_by_counts", "Genes detected per cell"),
    ("total_counts",      "Total UMI counts per cell"),
    ("pct_counts_mt",     "Mitochondrial gene fraction (%)"),
]
for ax, (metric, title) in zip(axes, metrics):
    sc.pl.violin(adata_filt, metric, groupby="sample_id", rotation=45, show=False, ax=ax, size=0.3)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel("")
plt.suptitle("QC distributions per sample (after filtering)", fontsize=12, y=1.01)
plt.tight_layout()
fig.savefig(FIG_DIR / "qc_violin_after_filter.png", dpi=200, bbox_inches="tight")
plt.show()

## Step 6 — Preserve Raw Counts and Save

Raw integer counts must be preserved in `.layers['counts']` because scVI (notebook 03) requires them as input. Once we normalize in notebook 02, `adata.X` will be overwritten with log-normalized values. This layer persists the original UMI counts through all downstream steps.

In [14]:
print("Storing raw counts in .layers['counts'] (required for scVI)")
adata_filt.layers["counts"] = adata_filt.X.copy()

# Save QC summary
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(TAB_DIR / "qc_summary_per_sample.csv", index=False)

# Save filtered atlas
print(f"\nSaving filtered atlas → {OUT_H5AD.name}")
adata_filt.write_h5ad(OUT_H5AD, compression="gzip")
size_mb = OUT_H5AD.stat().st_size / 1e6
print(f"  {adata_filt.n_obs:,} cells × {adata_filt.n_vars:,} genes | {size_mb:.0f} MB")
print(f"\n  Next: Run notebook 02_normalization_hvg.ipynb")

Storing raw counts in .layers['counts'] (required for scVI)

Saving filtered atlas → 02_qc_filtered.h5ad
  136,199 cells × 28,306 genes | 1479 MB

  Next: Run notebook 02_normalization_hvg.ipynb
